# Бенчмарк и выбор векторного хранилища

### Подготовка тестовых данных

In [1]:
import numpy as np
import time
from typing import List, Tuple

def generate_test_data(
    num_documents: int = 10000,
    dimension: int = 384,
    num_queries: int = 100
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Генерация тестовых данных для бенчмарка.
    
    Returns:
        vectors: Массив векторов документов
        queries: Массив векторов запросов
    """
    np.random.seed(42)  # Для воспроизводимости
    
    # Генерация векторов документов
    vectors = np.random.random((num_documents, dimension)).astype('float32')
    # Нормализация для косинусного расстояния
    vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    
    # Генерация запросов (похожих на документы для реалистичности)
    query_indices = np.random.choice(num_documents, num_queries, replace=False)
    queries = vectors[query_indices].copy()
    # Добавление небольшого шума
    queries += np.random.normal(0, 0.1, queries.shape).astype('float32')
    queries = queries / np.linalg.norm(queries, axis=1, keepdims=True)
    
    return vectors, queries

# Генерация данных
vectors, queries = generate_test_data(num_documents=10000, num_queries=100)
print(f"Создано {len(vectors)} векторов и {len(queries)} запросов")


Создано 10000 векторов и 100 запросов


## Бенчмарк FAISS

In [2]:
import faiss

def benchmark_faiss(vectors: np.ndarray, queries: np.ndarray, k: int = 5):
    """Бенчмарк FAISS."""
    dimension = vectors.shape[1]
    
    # 1. Создание индекса
    start_time = time.time()
    index = faiss.IndexFlatL2(dimension)
    index.add(vectors)
    index_time = time.time() - start_time
    
    # 2. Время одного запроса
    start_time = time.time()
    D, I = index.search(queries[:1], k)
    single_query_time = (time.time() - start_time) * 1000  # в миллисекундах
    
    # 3. Пропускная способность (батчевый поиск)
    start_time = time.time()
    D, I = index.search(queries, k)
    batch_time = time.time() - start_time
    qps = len(queries) / batch_time
    
    # 4. Использование памяти (приблизительно)
    memory_mb = (vectors.nbytes + len(queries) * k * 8) / (1024 * 1024)
    
    return {
        "name": "FAISS",
        "index_time": index_time,
        "single_query_ms": single_query_time,
        "qps": qps,
        "memory_mb": memory_mb
    }

# Запуск бенчмарка
faiss_results = benchmark_faiss(vectors, queries)
print("\n=== FAISS ===")
print(f"Время индексации: {faiss_results['index_time']:.3f} сек")
print(f"Время запроса: {faiss_results['single_query_ms']:.2f} мс")
print(f"Пропускная способность (батч): {faiss_results['qps']:.0f} QPS")
print(f"Память: {faiss_results['memory_mb']:.1f} МБ")


=== FAISS ===
Время индексации: 0.000 сек
Время запроса: 1.27 мс
Пропускная способность (батч): 50742 QPS
Память: 14.7 МБ


## Бенчмарк Qdrant

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

def benchmark_qdrant(vectors: np.ndarray, queries: np.ndarray, k: int = 5):
    """Бенчмарк Qdrant."""
    client = QdrantClient(url="http://localhost:6333")
    collection_name = "benchmark_test"
    dimension = vectors.shape[1]
    
    # Удаление коллекции если существует
    try:
        client.delete_collection(collection_name)
    except:
        pass
    
    # 1. Создание коллекции и индексация
    start_time = time.time()
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=dimension, distance=Distance.COSINE)
    )
    
    # Добавление векторов батчами для скорости
    batch_size = 1000
    points = [
        PointStruct(id=idx, vector=vector.tolist(), payload={})
        for idx, vector in enumerate(vectors)
    ]
    
    for i in range(0, len(points), batch_size):
        client.upsert(
            collection_name=collection_name,
            points=points[i:i + batch_size],
            wait=True
        )
    
    index_time = time.time() - start_time
    
    # 2. Время одного запроса (search() удалён, используем query_points())
    start_time = time.time()
    client.query_points(
        collection_name=collection_name,
        query=queries[0].tolist(),
        limit=k
    )
    single_query_time = (time.time() - start_time) * 1000
    
    # 3. Пропускная способность (последовательные запросы)
    start_time = time.time()
    for query in queries:
        client.query_points(
            collection_name=collection_name,
            query=query.tolist(),
            limit=k
        )
    batch_time = time.time() - start_time
    qps = len(queries) / batch_time
    
    # Очистка
    client.delete_collection(collection_name)
    
    return {
        "name": "Qdrant",
        "index_time": index_time,
        "single_query_ms": single_query_time,
        "qps": qps,
        "memory_mb": "N/A (disk-based)"
    }

# Запуск бенчмарка (требуется запущенный Qdrant)
try:
    qdrant_results = benchmark_qdrant(vectors, queries)
    print("\n=== Qdrant ===")
    print(f"Время индексации: {qdrant_results['index_time']:.3f} сек")
    print(f"Время запроса: {qdrant_results['single_query_ms']:.2f} мс")
    print(f"Пропускная способность: {qdrant_results['qps']:.0f} QPS")
    print(f"Память: {qdrant_results['memory_mb']}")
except Exception as e:
    print(f"\n=== Qdrant ===")
    print(f"Ошибка: {e}")
    print("Убедитесь, что Qdrant запущен: docker run -p 6333:6333 qdrant/qdrant")

| Метрика | FAISS (in-process) | Qdrant (HTTP) | Weaviate (HTTP) |
| :--- | :--- | :--- | :--- |
| **Время индексации** | ~0.1–0.5 сек [web:10] | ~2–5 сек [web:12] | ~3–8 сек [web:14] |
| **Latency (1 запрос)** | ~1–3 мс [web:10] | ~5–15 мс [web:12] | ~10–30 мс [web:14] |
| **QPS (тест)** | ~500–1000+ [web:10] | ~100–300 [web:12] | ~50–150 [web:14] |
| **Память (10K)** | ~15 МБ в RAM [web:10] | Диск + кэш [web:12] | Диск + кэш [web:14] |
| **Масштабируемость** | До миллионов [web:10] | Миллиарды (диск) [web:12] | Миллиарды (кластер) [web:14] |

Особенности представленных систем

- FAISS: Является библиотекой, а не полноценной базой данных, что обеспечивает минимальные задержки за счет работы напрямую в оперативной памяти. Он идеален для случаев, когда важна максимальная скорость поиска, а данные умещаются в RAM.
- Qdrant: Написан на Rust, что делает его одной из самых быстрых полноценных БД с поддержкой расширенной фильтрации метаданных. Он хорошо балансирует между скоростью и возможностью масштабирования на диске.
- Weaviate: Фокусируется на удобстве использования в корпоративной среде и интеграции с графами знаний, используя GraphQL для запросов. Хотя его задержки выше, он предлагает наиболее богатый функционал для работы со сложными связями в данных.